# Step 2. 기준선(Baseline) 잡기 & 함정 1 — 점수 100점짜리 가짜 모델

- **예상 소요 시간:** 약 40분
- **목표:**
  1. 머신러닝 모델 성능의 바닥 기준선(**Baseline**)과 채점 기준(**MAE**)을 잡는다.
  2. Step 1에서 본 `casual`, `registered` 컬럼을 **일부러 안 지우고 학습시켜** 오차 `0.00대`짜리 가짜 모델 버그(**데이터 누수, Data Leakage**)를 직접 터뜨린다.

---

### 백엔드 개발자 관점으로 이해하기
API 성능 튜닝을 할 때 "응답속도 빨라졌어요!"라고 말하면 바로 까인다.  
**"튜닝 전 p95가 800ms였고, 인덱스 추가 후 120ms로 줄었다"**처럼 **기준선(Before)** 숫자가 있어야 한다.

머신러닝도 똑같다.
- **Baseline (기준선):** 가장 멍청한 예측 방법("무조건 과거 평균값으로 찍기")의 오차
- **MAE (Mean Absolute Error):** 실제값과 예측값 차이의 절댓값 평균.  
  우리 문제에서는 **"평균적으로 자전거 몇 대 틀렸는가"**를 뜻한다. 숫자가 작을수록 좋다.

---
## 1. 데이터 준비 & `X`(입력) / `y`(정답) 나누기

백엔드 API로 치면 **`X`는 Request DTO**, **`y`는 Response(정답)**다.  
우선 데이터를 불러온 뒤, 일부러 함정을 밟기 위해 `casual`과 `registered`까지 **전부** `X`에 넣어본다.

In [2]:
import os
import urllib.request
import zipfile
import numpy as np                  # 수치·배열 연산 라이브러리 (관례적 별칭 np)
import pandas as pd                 # 표(DataFrame) 데이터 처리 라이브러리
import matplotlib.pyplot as plt     # 그래프 시각화 라이브러리

# 1) dataset/hour.csv가 없으면 UCI 공식 저장소에서 자동 다운로드 (로컬 + Colab 공용)
os.makedirs("dataset", exist_ok=True)  # 폴더가 없으면 생성, 이미 있으면 에러 없이 통과 (Java Files.createDirectories와 동일)
if not os.path.exists("dataset/hour.csv"):
    url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
    zip_path = "dataset/bike_sharing_dataset.zip"
    print("데이터 다운로드 중...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("dataset")
    print("다운로드 완료!")

df = pd.read_csv("dataset/hour.csv")
print(f"로드 완료: {df.shape[0]:,}행 × {df.shape[1]}컬럼")

로드 완료: 17,379행 × 17컬럼


### 🔍 코드 뜯어보기
- **`os.makedirs("dataset", exist_ok=True)`:**  
  `dataset` 폴더가 없으면 새로 만들고, 이미 있으면(`exist_ok=True`) 에러 내지 말고 그냥 넘어가라는 옵션이다. Java의 `Files.createDirectories(...)`와 같다.
- **로컬 + Colab 공용 패턴:**  
  로컬 `regression/` 폴더에서는 이미 `dataset/hour.csv`가 있으니 다운로드를 건너뛰고, Google Colab에 이 노트북만 단독 업로드해도 자동으로 받아서 바로 돌아간다.

---
## 2. 바닥 점수(Baseline) 재기 — "무조건 평균값으로 찍으면 몇 대 틀릴까?"

> **[실행 전 예측]**  
> `hour.csv` 전체 시간대 평균 대여량은 약 189대다.  
> 새벽 3시든, 평일 출근 8시든, 비가 오든 말든 **무조건 "189대!"**라고 찍으면 평균적으로 몇 대쯤 틀릴까? (50대? 140대? 300대?)

In [3]:
from sklearn.model_selection import train_test_split                   # 학습/테스트 데이터 분할 함수
from sklearn.metrics import mean_absolute_error, root_mean_squared_error  # 오차 채점 지표 (MAE, RMSE)

# 학습할 입력 컬럼 목록 (일부러 함정 컬럼 casual, registered 포함!)
cols_with_leak = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed",
    "casual", "registered"  # <-- 함정! (둘을 더하면 정답 cnt가 그대로 나옴)
]

X_leak = df[cols_with_leak]  # 관례상 입력 피처는 대문자 X (2차원 표, DataFrame — sklearn 필수 형식)
y = df["cnt"]                # 관례상 정답(타겟)은 소문자 y (1차원 목록, Series)

# 전체 17,379행을 학습용 80% / 테스트용 20%로 분리 (튜플 언패킹으로 4개 변수 한 번에 수신)
# random_state=42: 랜덤 시드 고정 (다시 실행해도 똑같이 분할되어 공정한 실험 비교 가능)
X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.2, random_state=42
)

# 바닥 모델(Baseline): 학습 데이터의 평균 대여량 1개 숫자로 무조건 찍기
train_mean = y_train.mean()
# np.full: y_test 길이만큼 train_mean 값으로 채운 배열 생성 (Java Arrays.fill과 동일)
baseline_pred = np.full(shape=len(y_test), fill_value=train_mean)

# 채점: MAE는 평균 절대 오차(직관적 대수), RMSE는 큰 오차에 가중 페널티
base_mae = mean_absolute_error(y_test, baseline_pred)
base_rmse = root_mean_squared_error(y_test, baseline_pred)

print(f"학습 데이터 평균 대여량: {train_mean:.1f}대")
print(f"[Baseline] 무조건 평균 찍기 → MAE: {base_mae:.2f}대 (RMSE: {base_rmse:.2f}대)")

학습 데이터 평균 대여량: 190.6대
[Baseline] 무조건 평균 찍기 → MAE: 140.08대 (RMSE: 178.03대)


### 🔍 코드 뜯어보기
- **`from sklearn.model_selection import train_test_split`:**  
  `scikit-learn`(임포트 이름은 `sklearn`)은 파이썬 머신러닝의 표준 라이브러리다. Spring Boot의 기본 스타터 같은 존재다.
- **`X_leak = df[cols_with_leak]` vs `y = df["cnt"]`:**  
  - 관례상 입력 데이터는 대문자 **`X`**(2차원 표, DataFrame), 정답은 소문자 **`y`**(1차원 목록, Series)로 쓴다.
  - `sklearn`의 모든 모델은 **`X`가 2차원 표**라고 가정한다.
- **`train_test_split(X_leak, y, test_size=0.2, random_state=42)`:**  
  - 전체 17,379행을 학습용 80%(13,903행)와 시험용 20%(3,476행)로 쪼개 4개 변수를 한 번에 반환한다. (파이썬 튜플 언패킹)
  - `random_state=42`: 랜덤 시드 고정. 이 숫자를 고정해야 나중에 다시 실행해도 똑같이 쪼개져 실험 비교가 된다. (42는 개발자 세계 관례 숫자)
- **`np.full(shape=len(y_test), fill_value=train_mean)`:**  
  `y_test` 길이(3,476)만큼 `train_mean` 숫자로 꽉 채운 배열을 만든다. Java의 `Arrays.fill(...)`과 같다.
- **`mean_absolute_error(y_test, baseline_pred)`:**  
  `(실제값 - 예측값)`의 절댓값을 3,476개 전부 더해 평균을 낸다.  
  *참고:* `RMSE`는 오차를 제곱해서 평균낸 뒤 루트를 씌운 값이라, **크게 틀린 몇 건에 페널티를 세게 준다.** 직관적인 대수 파악은 `MAE`가 훨씬 쉽다.

### 결과 해석
- 아무것도 안 배우고 무조건 "189대!"라고 찍으면 평균 **`140.08대`** 틀린다.
- 이제부터 우리가 만드는 어떤 모델이든 **최소한 140대보다는 훨씬 적게 틀려야** 쓸모가 있다.

---
## 3. 함정 1 터뜨리기 — 모든 컬럼 그대로 넣고 첫 모델 학습

> **[실행 전 예측]**  
> 이제 `LinearRegression`(선형회귀) 모델에 `X_train`을 넣고 학습시킨 뒤 테스트 점수를 잰다.  
> 방금 `casual`과 `registered`를 일부러 안 지웠다. 오차(MAE)가 대략 몇 대쯤 나올까?

In [5]:
from sklearn.linear_model import LinearRegression  # 선형회귀 모델 (일차방정식 수식 학습)

# sklearn 모델 3단계 공식: 객체 생성 → .fit(학습) → .predict(예측) (Java 인터페이스 다형성과 동일)
# 1) 모델 생성 및 학습 (빌드 타임: 과거 X_train과 정답 y_train으로 규칙 수식 생성)
model_leak = LinearRegression()
model_leak.fit(X_train, y_train)

# 2) 테스트 데이터로 예측 (런타임: 정답을 가린 X_test만 입력해 예측값 산출)
pred_leak = model_leak.predict(X_test)

# 3) 채점: MAE(평균 몇 대 틀렸나) & R²(1.0이면 완벽 예측, 0.0이면 평균 찍기 수준)
mae_leak = mean_absolute_error(y_test, pred_leak)
r2_leak = model_leak.score(X_test, y_test)

print(f"[함정 1: 누수 포함] MAE: {mae_leak:.6f}대, R²(설명력): {r2_leak:.6f}")

# 4) 모델이 각 컬럼에 붙인 가중치(계수) 뜯어보기
# model_leak.coef_: sklearn에서는 .fit() 이후 생성되는 학습 결과 속성에 항상 끝 언더스코어(_)를 붙임
# pd.Series(..., index=...): 숫자 배열에 컬럼명을 키로 붙여 보기 좋게 매핑
coef_df = pd.Series(model_leak.coef_, index=cols_with_leak).round(4)
print("\n[모델이 학습한 컬럼별 가중치(coef)]")
print(coef_df)

[함정 1: 누수 포함] MAE: 0.000000대, R²(설명력): 1.000000

[모델이 학습한 컬럼별 가중치(coef)]
season       -0.0
yr           -0.0
mnth          0.0
hr           -0.0
holiday      -0.0
weekday       0.0
workingday   -0.0
weathersit    0.0
temp          0.0
atemp        -0.0
hum           0.0
windspeed    -0.0
casual        1.0
registered    1.0
dtype: float64


### 🔍 코드 뜯어보기
- **`sklearn` 모델 사용법은 어떤 모델이든 3줄 고정이다** (Java 인터페이스의 다형성과 똑같다!):
  1. `model = LinearRegression()` — 객체 생성
  2. `model.fit(X_train, y_train)` — **학습(규칙 찾기)**. 과거 입력(`X_train`)과 정답(`y_train`)을 보고 수식을 만든다.
  3. `pred = model.predict(X_test)` — **추론(예측)**. 정답을 가린 `X_test`만 넘겨서 예측값을 받아온다.
- **`model.score(X_test, y_test)`:**  
  회귀 모델에서 기본 `score()`는 **R²(결정계수)**를 돌려준다. `1.0`이면 100% 완벽 예측, `0.0`이면 평균 찍기 수준이라는 뜻이다.
- **`model_leak.coef_`:**  
  선형회귀는 `예측값 = (가중치1 × 컬럼1) + (가중치2 × 컬럼2) + ... + 절편` 형태의 수식을 만든다.  
  `coef_` 뒤에 언더스코어(`_`)가 붙어 있는데, `sklearn`에서는 **"`.fit()`을 실행한 뒤에야 생기는 학습 결과 속성"**에 항상 끝에 `_`를 붙인다.
- **`pd.Series(model_leak.coef_, index=cols_with_leak)`:**  
  숫자 배열만 찍으면 어느 컬럼의 가중치인지 못 알아보므로, 컬럼 이름을 인덱스(키)로 붙여서 보기 좋게 만든다.

### 결과 해석: 이게 바로 '데이터 누수(Data Leakage)' 버그다!
출력을 보자.
- **MAE = `0.000000대`**, **R² = `1.000000`** (100점 만점에 100점!)
- 가중치(`coef`)를 보면 충격적이다.
  - `casual`: **`1.0000`**
  - `registered`: **`1.0000`**
  - 나머지 기온, 시간, 날씨, 계절: **전부 `0.0000`!**

모델이 날씨와 시간 패턴을 배운 게 아니다.  
**"`cnt = 1.0 × casual + 1.0 × registered`네? 나머지 다 무시하고 둘만 더하자!"** 하고 덧셈 계산기가 되어버렸다.

**Spring 개발자 번역:**  
단위 테스트에 정답 객체를 미리 Mock으로 박아두고 "테스트 100% 통과!" 하고 좋아하다가 운영 배포 직후 장애 내는 버그와 100% 똑같다.

> **실무 황금 법칙:**  
> 처음 돌린 모델이 **너무 말이 안 되게 잘 나오면(오차 0, 정확도 99.9%)** 천재가 된 게 아니다. **100% 데이터 누수 버그**부터 의심해라.

---
## 4. 버그 수정 — `casual`, `registered` 제거 후 정상 점수 확인

이제 운영 API 요청 JSON에 실제로 담을 수 있는 12개 컬럼만 남기고 다시 돌려보자.

In [6]:
# 운영 API 요청 시점에 실제로 알 수 있는 입력 컬럼 12개만 사용 (casual, registered 제거)
clean_cols = [
    "season", "yr", "mnth", "hr", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed"
]

X_clean = df[clean_cols]
# 같은 시드(random_state=42)로 80:20 분할해 앞선 실험과 동일 조건 유지
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clean, y, test_size=0.2, random_state=42
)

# 모델 생성 → 학습 → 예측 (동일한 3줄 패턴)
model_clean = LinearRegression()
model_clean.fit(Xc_train, yc_train)
pred_clean = model_clean.predict(Xc_test)

# 오차(MAE) 및 설명력(R²) 계산
mae_clean = mean_absolute_error(yc_test, pred_clean)
r2_clean = model_clean.score(Xc_test, yc_test)

print(f"[Baseline] 무조건 평균 찍기 → MAE: {base_mae:.2f}대")
print(f"[버그 수정] 누수 제거 선형회귀 → MAE: {mae_clean:.2f}대 (R²: {r2_clean:.3f})")

# 실제값 vs 예측값 5건만 즉석 DataFrame(딕셔너리 → 표)으로 눈으로 비교
# yc_test.values[:5]: 원래 행 인덱스(12830 등)를 떼고 순수 숫자 배열로 꺼낸 뒤 앞 5개 슬라이싱
compare_df = pd.DataFrame({
    "actual(cnt)": yc_test.values[:5],
    "predicted": pred_clean[:5].round(1),                    # 예측값 중 음수(-)가 나오는 한계 확인!
    "error": (yc_test.values[:5] - pred_clean[:5]).round(1)  # 실제값 - 예측값
})
print("\n[실제값 vs 예측값 샘플 5건]")
print(compare_df)

[Baseline] 무조건 평균 찍기 → MAE: 140.08대
[버그 수정] 누수 제거 선형회귀 → MAE: 104.80대 (R²: 0.388)

[실제값 vs 예측값 샘플 5건]
   actual(cnt)  predicted  error
0          425      450.3  -25.3
1           88      204.7 -116.7
2            4       57.3  -53.3
3          526      383.4  142.6
4           13      -20.3   33.3


### 🔍 코드 뜯어보기
- **`pd.DataFrame({...})`:**  
  파이썬 딕셔너리(`{"컬럼명": 배열}`)를 넘겨서 즉석에서 비교용 표를 만든다.
- **`yc_test.values[:5]`:**  
  `yc_test`는 원래 데이터프레임의 행 인덱스(예: 12830, 8688...)를 그대로 들고 있다. `.values`로 순수 숫자 배열만 꺼낸 뒤 앞 5개(`[:5]`)를 잘라 `pred_clean[:5]`와 나란히 맞춘다.
- **잠깐! `predicted` 중에 음수(`-`)가 보이나요?**  
  선형회귀는 그냥 직선 수식이라, 대여량이 적은 새벽 시간대에는 **"-20대"** 같은 현실에 없는 음수 예측도 뱉는다. 이것도 선형회귀의 한계 중 하나다.

---
## 5. 직접 해보는 자습 미션 (10분)

- **미션 1:** `model_clean.predict(Xc_test)` 결과 중 **0보다 작은 음수 예측**이 몇 건이나 나왔는지 세어보자.  
  *(힌트: `(pred_clean < 0).sum()`)*
- **미션 2:** 음수 예측을 전부 `0`으로 잘라주면(`np.clip(pred_clean, 0, None)`) MAE가 `104.80대`에서 얼마나 줄어들까?

In [7]:
# [미션 1 & 2] 음수 예측 건수 확인 & 0으로 클리핑 후 MAE 재측정
# TODO: 아래에 코드를 작성해 보세요.

# 미션 1: 음수 예측 건수 세기 ((pred_clean < 0)의 True=1 개수를 .sum()으로 합산)
neg_count = int((pred_clean < 0).sum())
print(f"음수로 예측된 건수: {neg_count}건 / 전체 {len(pred_clean)}건")

# 미션 2: np.clip(배열, 최소값, 최대값)으로 0 미만 음수를 전부 0으로 자르기 (Java Math.max(0, val)의 배열 버전)
pred_clipped = np.clip(pred_clean, 0, None)  # 최대값 None = 상한선 제한 없음
mae_clipped = mean_absolute_error(yc_test, pred_clipped)
print(f"클리핑 전 MAE: {mae_clean:.2f}대 → 클리핑 후 MAE: {mae_clipped:.2f}대")

음수로 예측된 건수: 160건 / 전체 3476건
클리핑 전 MAE: 104.80대 → 클리핑 후 MAE: 103.36대


<details>
<summary>정답 코드 & 문법 해설 보기 (먼저 직접 쳐본 뒤 열어보세요)</summary>

```python
# 미션 1: 음수 예측 건수
neg_count = int((pred_clean < 0).sum())
print(f"음수로 예측된 건수: {neg_count}건 / 전체 {len(pred_clean)}건")

# 미션 2: 음수를 0으로 자르기 (하한선 0, 상한선 없음)
pred_clipped = np.clip(pred_clean, 0, None)
mae_clipped = mean_absolute_error(yc_test, pred_clipped)
print(f"클리핑 전 MAE: {mae_clean:.2f}대 → 클리핑 후 MAE: {mae_clipped:.2f}대")
```

- **`np.clip(배열, 최소값, 최대값)`:** 배열의 숫자를 지정한 범위 안으로 가둔다. `최대값`에 `None`을 넣으면 위쪽 제한 없이 `0 미만`만 전부 `0`으로 바꾼다. Java의 `Math.max(0, val)`을 배열 전체에 한 번에 적용한 것이다.
- 도메인 상식("자전거 대여량은 절대 음수가 될 수 없다") 한 줄만 얹어도 오차가 공짜로 줄어든다.
</details>

---
## 6. 30분 발표용 정리 (Step 2 분량 = 발표에서 약 3~4분)

### ① 내 실험 기록표에 적을 숫자
- `Exp 0 (Baseline 평균 찍기, 랜덤 분할)`: MAE **140.08대**
- `Exp 1 (함정 1: 누수 포함 선형회귀)`: MAE **0.00대** (R² = 1.000)
- `Exp 2a (누수 제거 선형회귀, 랜덤 분할)`: MAE **104.80대**

### ② 발표 대본 샘플 (그대로 읽어도 자연스러운 구어체)
> "첫 번째로 일부러 밟아본 함정이 **데이터 누수**입니다. 먼저 기준선을 잡으려고 '무조건 평균 189대'라고 찍어봤더니 평균 140대 틀렸습니다.
>
> 그다음에 `casual`이랑 `registered`를 안 지우고 선형회귀를 돌려봤더니 오차가 **0.00대, R²가 1.000**이 나왔습니다. 처음엔 와 천재인가 싶었는데요. 모델 가중치를 까보니까 `casual`에 1, `registered`에 1을 곱하고 날씨랑 시간은 전부 0으로 무시하고 있었습니다. 그냥 덧셈 계산기가 된 거죠.
>
> 백엔드로 치면 테스트 코드에 정답을 미리 Mock으로 박아두고 단위 테스트 통과했다고 착각한 버그입니다. 두 컬럼을 지우고 운영 API에서 실제로 받을 수 있는 12개 컬럼만 남겼더니 오차가 104.8대로 현실적인 숫자가 나왔습니다.
>
> 근데 여기서 끝이 아니었습니다. 다음 **Step 3**에서 훨씬 교묘한 두 번째 함정을 만났습니다."

---
**Step 2 완료!**  
다음 **Step 3**에서는 "점수가 24대까지 떨어졌다!"고 좋아했다가 뒤통수 맞는 **함정 2(시계열 랜덤 분할 컨닝)**를 직접 체험한다.